# 03 Búsqueda vectorial con TF-IDF

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

In [8]:
import re
from collections import Counter
from pathlib import Path
import nltk
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from nltk.corpus import gutenberg
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    file_ids = sorted(gutenberg.fileids())
except LookupError:
    ruta_datos = Path.cwd() / 'nltk_data'
    nltk.data.path.insert(0, str(ruta_datos))
    if not nltk.download('gutenberg', download_dir=str(ruta_datos), quiet=True):
        raise RuntimeError('No se pudo descargar Gutenberg. Revise la conexión.')
    file_ids = sorted(gutenberg.fileids())

def normalizar_y_tokenizar(texto):
    texto = texto.lower()
    texto = re.sub(r'[^a-záéíóúñü]+', ' ', texto)
    return texto.split()

raw_docs = {fid: gutenberg.raw(fid) for fid in file_ids}
documentos_tokenizados = {
    fid: normalizar_y_tokenizar(raw_docs[fid]) for fid in file_ids
}
def obtener_titulo(texto):
    encabezado = re.search(r'\[([^\]]+)\]', texto)
    return encabezado.group(1).strip() if encabezado else texto.splitlines()[0].strip()

titulos = {fid: obtener_titulo(raw_docs[fid]) for fid in file_ids}
tokens_corpus = [documentos_tokenizados[fid] for fid in file_ids]
N = len(file_ids)
longitudes = np.array([len(tokens) for tokens in tokens_corpus])
tabla_corpus = pd.DataFrame({
    'Identificador de documento': file_ids,
    'Título': [titulos[fid] for fid in file_ids],
    'Número de términos únicos': [len(set(tokens)) for tokens in tokens_corpus]
})
display(tabla_corpus)
vocabulario_corpus = set().union(*(set(tokens) for tokens in tokens_corpus))
print(f'Número de documentos: {N}')
print(f'Tamaño del vocabulario: {len(vocabulario_corpus):,}')
print(f'Longitud mínima: {longitudes.min():,} tokens')
print(f'Longitud media: {longitudes.mean():,.2f} tokens')
print(f'Longitud máxima: {longitudes.max():,} tokens')
assert N == 18

,Identificador de documento,Título,Número de términos únicos
0,austen-emma.txt,Emma by Jane Austen 1816,7096
1,austen-persuasion.txt,Persuasion by Jane Austen 1818,5740
2,austen-sense.txt,Sense and Sensibility by Jane Austen 1811,6283
3,bible-kjv.txt,The King James Bible,12568
4,blake-poems.txt,Poems by William Blake 1789,1510
5,bryant-stories.txt,Stories to Tell to Children by Sara Cone Bryan...,3834
6,burgess-busterbrown.txt,The Adventures of Buster Bear by Thornton W. B...,1521
7,carroll-alice.txt,Alice's Adventures in Wonderland by Lewis Carr...,2569
8,chesterton-ball.txt,The Ball and The Cross by G.K. Chesterton 1909,8177
9,chesterton-brown.txt,The Wisdom of Father Brown by G. K. Chesterton...,7737


Número de documentos: 18
Tamaño del vocabulario: 41,509
Longitud mínima: 6,934 tokens
Longitud media: 118,671.06 tokens
Longitud máxima: 791,842 tokens


### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

In [9]:
def calcular_tf(tokens):
    """TF(t,d): número de apariciones del término en el documento."""
    return Counter(tokens)

def calcular_df(documentos):
    """Cada documento aporta como máximo una aparición por término."""
    frecuencias = Counter()
    for tokens in documentos:
        frecuencias.update(set(tokens))
    return frecuencias

def calcular_idf(df, numero_documentos):
    """IDF natural sin suavizado: ln(N / DF) + 1."""
    return {termino: np.log(numero_documentos / frecuencia) + 1.0
            for termino, frecuencia in df.items()}

def calcular_tfidf(tf, idf):
    return {termino: frecuencia * idf[termino]
            for termino, frecuencia in tf.items() if termino in idf}

ejemplo_docs = [['love', 'love', 'sea'], ['love', 'whale', 'whale']]
ejemplo_df = calcular_df(ejemplo_docs)
ejemplo_idf = calcular_idf(ejemplo_df, len(ejemplo_docs))
filas_ejemplo = []
for numero, tokens in enumerate(ejemplo_docs, 1):
    tf = calcular_tf(tokens)
    pesos = calcular_tfidf(tf, ejemplo_idf)
    for termino in ['love', 'sea', 'whale']:
        filas_ejemplo.append({
            'Documento': f'D{numero}', 'Término': termino,
            'TF': tf[termino], 'DF': ejemplo_df[termino],
            'IDF': ejemplo_idf[termino], 'TF-IDF': pesos.get(termino, 0.0)
        })
display(pd.DataFrame(filas_ejemplo))
assert len(filas_ejemplo) == 6

,Documento,Término,TF,DF,IDF,TF-IDF
0,D1,love,2,2,1.000000,2.000000
1,D1,sea,1,1,1.693147,1.693147
2,D1,whale,0,1,1.693147,0.000000
3,D2,love,1,2,1.000000,1.000000
4,D2,sea,0,1,1.693147,0.000000
5,D2,whale,2,1,1.693147,3.386294


**Interpretación de los seis pesos.** Se presentan tres términos por dos documentos, pues el enunciado no identifica seis pares concretos. `love` tiene DF = 2 e IDF = 1; su peso coincide con su frecuencia. `sea` y `whale` tienen DF = 1 e IDF = ln(2) + 1: son más discriminativos. Un término ausente tiene TF y peso cero. Para el corpus real se calculan después todos los pesos documento–término.

### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

In [10]:
vocabulario = sorted(vocabulario_corpus)
indice_terminos = {termino: j for j, termino in enumerate(vocabulario)}
tf_documentos = [calcular_tf(tokens) for tokens in tokens_corpus]
df_corpus = calcular_df(tokens_corpus)
idf_corpus = calcular_idf(df_corpus, N)
vector_idf = np.array([idf_corpus[t] for t in vocabulario])

matriz_tf = np.zeros((N, len(vocabulario)), dtype=np.float64)
for i, frecuencias in enumerate(tf_documentos):
    for termino, frecuencia in frecuencias.items():
        matriz_tf[i, indice_terminos[termino]] = frecuencia
matriz_tfidf = matriz_tf * vector_idf

assert np.array_equal(np.count_nonzero(matriz_tf, axis=0),
                      np.array([df_corpus[t] for t in vocabulario]))
assert np.array_equal(matriz_tf.sum(axis=1), longitudes)
print('Dimensiones documento-término:', matriz_tfidf.shape)
terminos_inspeccion = ['the', 'love', 'sea', 'whale', 'god', 'hamlet']
display(pd.DataFrame({
    'Término': terminos_inspeccion,
    'DF': [df_corpus[t] for t in terminos_inspeccion],
    'IDF': [idf_corpus[t] for t in terminos_inspeccion]
}))
columnas = [indice_terminos[t] for t in terminos_inspeccion]
display(pd.DataFrame(matriz_tfidf[:, columnas],
                     index=file_ids, columns=terminos_inspeccion))

Dimensiones documento-término: (18, 41509)


,Término,DF,IDF
0,the,18,1.000000
1,love,15,1.182322
2,sea,16,1.117783
3,whale,6,2.098612
4,god,16,1.117783
5,hamlet,6,2.098612


,the,love,sea,whale,god,hamlet
austen-emma.txt,5204.0,138.331622,15.648962,0.000000,8.942264,0.000000
austen-persuasion.txt,3329.0,49.657505,23.473444,0.000000,11.177830,0.000000
austen-sense.txt,4105.0,91.038760,0.000000,0.000000,11.177830,2.098612
bible-kjv.txt,64023.0,367.702004,447.113214,6.295837,4998.725735,0.000000
blake-poems.txt,439.0,34.287325,1.117783,0.000000,12.295613,0.000000
bryant-stories.txt,3452.0,15.370180,22.355661,33.577797,30.180142,0.000000
burgess-busterbrown.txt,660.0,1.182322,0.000000,0.000000,0.000000,0.000000
carroll-alice.txt,1642.0,3.546965,15.648962,0.000000,0.000000,0.000000
chesterton-ball.txt,4981.0,29.558039,63.713633,4.197225,156.489625,4.197225
chesterton-brown.txt,4670.0,10.640894,69.302548,0.000000,25.709010,0.000000


### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

In [11]:
def identidad(tokens):
    return tokens
vectorizador = TfidfVectorizer(
    analyzer=identidad,
    vocabulary=indice_terminos,
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm=None,
    dtype=np.float64
)
matriz_sklearn = vectorizador.fit_transform(tokens_corpus)
TOLERANCIA = 1e-10
diferencia_maxima = np.max(np.abs(matriz_tfidf - matriz_sklearn.toarray()))
print(f'Máxima diferencia absoluta TF-IDF: {diferencia_maxima:.12g}')
print(f'Tolerancia absoluta: {TOLERANCIA}')
assert list(vectorizador.get_feature_names_out()) == vocabulario
np.testing.assert_allclose(vectorizador.idf_, vector_idf, rtol=0, atol=TOLERANCIA)
np.testing.assert_allclose(matriz_tfidf, matriz_sklearn.toarray(),
                           rtol=0, atol=TOLERANCIA)
print('Verificación correcta: coinciden el vocabulario, los IDF y los pesos.')

Máxima diferencia absoluta TF-IDF: 0
Tolerancia absoluta: 1e-10
Verificación correcta: coinciden el vocabulario, los IDF y los pesos.


### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

In [12]:
consultas = [
    {'id': 'Q1', 'texto': 'whale sea ship',
     'necesidad': 'Encontrar relatos de navegación y caza de ballenas.'},
    {'id': 'Q2', 'texto': 'love marriage family',
     'necesidad': 'Encontrar historias de relaciones amorosas, matrimonio y familia.'},
    {'id': 'Q3', 'texto': 'god heaven faith',
     'necesidad': 'Encontrar textos sobre religión, fe y vida espiritual.'},
    {'id': 'Q4', 'texto': 'king murder ghost',
     'necesidad': 'Encontrar historias de reyes, asesinatos y apariciones fantasmales.'}
]
display(pd.DataFrame(consultas))
tokens_consultas = [normalizar_y_tokenizar(q['texto']) for q in consultas]
vectores_consultas = np.zeros((len(consultas), len(vocabulario)))
for i, tokens in enumerate(tokens_consultas):
    for termino, peso in calcular_tfidf(calcular_tf(tokens), idf_corpus).items():
        vectores_consultas[i, indice_terminos[termino]] = peso
    print(consultas[i]['id'], 'tokens:', tokens,
          '| Fuera del vocabulario:', sorted(set(tokens) - set(indice_terminos)))
consultas_sklearn = vectorizador.transform(tokens_consultas)
np.testing.assert_allclose(vectores_consultas, consultas_sklearn.toarray(),
                           rtol=0, atol=TOLERANCIA)

def similitud_coseno_explicita(consultas_matriz, documentos_matriz):
    productos = consultas_matriz @ documentos_matriz.T
    normas_q = np.sqrt(np.sum(consultas_matriz ** 2, axis=1))
    normas_d = np.sqrt(np.sum(documentos_matriz ** 2, axis=1))
    denominadores = normas_q[:, None] * normas_d[None, :]
    return np.divide(productos, denominadores,
                     out=np.zeros_like(productos), where=denominadores != 0)

similitudes = similitud_coseno_explicita(vectores_consultas, matriz_tfidf)
similitudes_sklearn = cosine_similarity(consultas_sklearn, matriz_sklearn)
error_coseno = np.max(np.abs(similitudes - similitudes_sklearn))
print(f'Máxima diferencia absoluta de coseno: {error_coseno:.12g}')
np.testing.assert_allclose(similitudes, similitudes_sklearn,
                           rtol=0, atol=TOLERANCIA)

i_q = 0
i_d = int(np.argmax(similitudes[i_q]))
q = vectores_consultas[i_q]
d = matriz_tfidf[i_d]
compartidos = np.flatnonzero((q != 0) & (d != 0))
print('Desglose:', consultas[i_q]['id'], '|', file_ids[i_d], '|', titulos[file_ids[i_d]])
display(pd.DataFrame({
    'Términos compartidos': [vocabulario[j] for j in compartidos],
    'Peso consulta': q[compartidos], 'Peso documento': d[compartidos],
    'Productos parciales': q[compartidos] * d[compartidos]
}))
display(pd.DataFrame([{
    'Producto punto': float(q @ d),
    'Norma consulta': float(np.sqrt(np.sum(q ** 2))),
    'Norma documento': float(np.sqrt(np.sum(d ** 2))),
    'Similitud': similitudes[i_q, i_d]
}]))
# Una consulta vacía o con todos los términos desconocidos obtiene coseno cero.
assert np.all(similitud_coseno_explicita(np.zeros((1, len(vocabulario))),
                                        matriz_tfidf) == 0)

,id,texto,necesidad
0,Q1,whale sea ship,Encontrar relatos de navegación y caza de ball...
1,Q2,love marriage family,"Encontrar historias de relaciones amorosas, ma..."
2,Q3,god heaven faith,"Encontrar textos sobre religión, fe y vida esp..."
3,Q4,king murder ghost,"Encontrar historias de reyes, asesinatos y apa..."


Q1 tokens: ['whale', 'sea', 'ship'] | Fuera del vocabulario: []
Q2 tokens: ['love', 'marriage', 'family'] | Fuera del vocabulario: []
Q3 tokens: ['god', 'heaven', 'faith'] | Fuera del vocabulario: []
Q4 tokens: ['king', 'murder', 'ghost'] | Fuera del vocabulario: []
Máxima diferencia absoluta de coseno: 2.74780198595e-15
Desglose: Q1 | melville-moby_dick.txt | Moby Dick by Herman Melville 1851


,Términos compartidos,Peso consulta,Peso documento,Productos parciales
0,sea,1.117783,508.591281,568.494706
1,ship,1.325422,686.568803,909.993671
2,whale,2.098612,2572.898666,5399.516758


,Producto punto,Norma consulta,Norma documento,Similitud
0,6878.005136,2.722197,21539.783392,0.117301


### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

In [13]:
def orden_ranking(puntuaciones):
    # Desempate reproducible mediante identificador original.
    return sorted(range(N), key=lambda i: (-float(puntuaciones[i]), file_ids[i]))

def obtener_fragmento(doc_id, tokens_consulta, ancho=220):
    texto = raw_docs[doc_id]
    patron = r'(?<!\w)(?:' + '|'.join(re.escape(t) for t in tokens_consulta) + r')(?!\w)'
    coincidencia = re.search(patron, texto, flags=re.IGNORECASE) if tokens_consulta else None
    inicio = max(0, coincidencia.start() - 65) if coincidencia else 0
    fin = min(len(texto), inicio + ancho)
    fragmento = re.sub(r'\s+', ' ', texto[inicio:fin]).strip()
    return ('…' if inicio else '') + fragmento + ('…' if fin < len(texto) else '')

def tabla_ranking(puntuaciones, tokens_consulta, top=5):
    filas = []
    for posicion, i in enumerate(orden_ranking(puntuaciones)[:top], 1):
        fid = file_ids[i]
        filas.append({'Posición': posicion, 'Identificador': fid,
                      'Título': titulos[fid], 'Puntuación': puntuaciones[i],
                      'Fragmento': obtener_fragmento(fid, tokens_consulta)})
    return pd.DataFrame(filas)

rankings_tfidf = {}
with pd.option_context('display.max_colwidth', 240):
    for i, consulta in enumerate(consultas):
        display(Markdown(f"**{consulta['id']}: {consulta['texto']}**"))
        display(Markdown(consulta['necesidad']))
        tabla = tabla_ranking(similitudes[i], tokens_consultas[i])
        rankings_tfidf[consulta['id']] = tabla
        display(tabla)
        orden_manual = orden_ranking(similitudes[i])[:5]
        orden_biblioteca = orden_ranking(similitudes_sklearn[i])[:5]
        assert orden_manual == orden_biblioteca
        np.testing.assert_allclose(similitudes[i], similitudes_sklearn[i],
                                   rtol=0, atol=TOLERANCIA)
        print('Top-5 y puntuaciones verificados con scikit-learn.')

**Q1: whale sea ship**

Encontrar relatos de navegación y caza de ballenas.

,Posición,Identificador,Título,Puntuación,Fragmento
0,1,melville-moby_dick.txt,Moby Dick by Herman Melville 1851,0.117301,"…take in hand to school others, and to teach them by what name a whale-fish is to be called in our tongue leaving out, through ignorance, the letter H, which almost alone maketh the signification of the word, you deliv…"
1,2,whitman-leaves.txt,Leaves of Grass by Walt Whitman 1855,0.010530,"…s, I above all promote brave soldiers. } In Cabin'd Ships at Sea In cabin'd ships at sea, The boundless blue on every side expanding, With whistling winds and music of the waves, the large imperious waves, Or some l…"
2,3,bryant-stories.txt,Stories to Tell to Children by Sara Cone Bryant 1918,0.007388,"…CLOUD One hot summer morning a little Cloud rose out of the sea and floated lightly and happily across the blue sky. Far below lay the earth, brown, dry, and desolate, from drought. The little Cloud could see the…"
3,4,chesterton-ball.txt,The Ball and The Cross by G.K. Chesterton 1909,0.006308,"…esterton 1909] I. A DISCUSSION SOMEWHAT IN THE AIR The flying ship of Professor Lucifer sang through the skies like a silver arrow; the bleak white steel of it, gleaming in the bleak blue emptiness of the evening. Th…"
4,5,chesterton-brown.txt,The Wisdom of Father Brown by G. K. Chesterton 1914,0.005437,"…ologist and specialist in certain moral disorders, lay along the sea-front at Scarborough, in a series of very large and well-lighted french windows, which showed the North Sea like one endless outer wall of blue-green m…"


Top-5 y puntuaciones verificados con scikit-learn.


**Q2: love marriage family**

Encontrar historias de relaciones amorosas, matrimonio y familia.

... (6 outputs left out)

**Q4: king murder ghost**

Encontrar historias de reyes, asesinatos y apariciones fantasmales.

,Posición,Identificador,Título,Puntuación,Fragmento
0,1,shakespeare-hamlet.txt,The Tragedie of Hamlet by William Shakespeare 1599,0.041966,"…Nay answer me: Stand & vnfold your selfe Bar. Long liue the King Fran. Barnardo? Bar. He Fran. You come most carefully vpon your houre Bar. 'Tis now strook twelue, get thee to bed Francisco Fran. F…"
1,2,shakespeare-macbeth.txt,The Tragedie of Macbeth by William Shakespeare 1603,0.018937,"…nd filthie ayre. Exeunt. Scena Secunda. Alarum within. Enter King Malcome, Donalbaine, Lenox, with attendants, meeting a bleeding Captaine. King. What bloody man is that? he can report, As seemeth by his plight, o…"
2,3,bryant-stories.txt,Stories to Tell to Children by Sara Cone Bryant 1918,0.014240,"…and in the jungle there was a great big Lion; and the Lion was king of the jungle. Whenever he wanted anything to eat, all he had to do was to come up out of his cave in the stones and earth and _roar_. When he had r…"
3,4,bible-kjv.txt,The King James Bible,0.014111,"[The King James Bible] The Old Testament of the King James Bible The First Book of Moses: Called Genesis 1:1 In the beginning God created the heaven and the earth. 1:2 And the earth was without form, and void; and…"
4,5,carroll-alice.txt,Alice's Adventures in Wonderland by Lewis Carroll 1865,0.011352,"…Alice, 'they're sure to kill it in a day or two: wouldn't it be murder to leave it behind?' She said the last words out loud, and the little thing grunted in reply (it had left off sneezing by this time). 'Don't grunt,'…"


Top-5 y puntuaciones verificados con scikit-learn.



### 7. Uso de inteligencia artificial

Se utilizó inteligencia artificial como apoyo para organizar el notebook, implementar las funciones de ponderación y recuperación, documentar el procedimiento y preparar las comprobaciones numéricas. Los enunciados del documento original se conservaron íntegros en sus celdas Markdown.